In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
import os
from pathlib import Path
from ultralytics import YOLO

In [3]:
base_dir = Path("../dataset")
images_dir = base_dir / "images"
labels_dir = base_dir / "labels"

train_images = images_dir / "train"
val_images = images_dir / "val"

In [4]:
yaml_path = base_dir / "data.yaml"
yaml_content = f"""
train: {train_images.resolve()}
val: {val_images.resolve()}

nc: 2
names: ['canceroso', 'benigno']
"""
yaml_path.write_text(yaml_content)
print(f"Archivo data.yaml creado en: {yaml_path}")

Archivo data.yaml creado en: ../dataset/data.yaml


In [ ]:
model = YOLO("yolov8n-seg.pt")

model.train(
    data=str(yaml_path),
    epochs=10,
    imgsz=512,
    batch=4,       # <--- INTENTO: Subimos de 4 a 8 (tienes memoria de sobra en la GPU)
    device=0,
    workers=1,     # Mantenemos esto por seguridad
    cache=False,    # <--- NUEVO: Cargar imágenes en RAM para acelerar
    name="colon_segmentation_yolov8",
    project="runs/segmentation"
)

New https://pypi.org/project/ultralytics/8.3.233 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.232 🚀 Python-3.10.12 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=colon_segmentation_yolov815, n

In [1]:
print("Generando predicciones de validación...")
results = model.predict(
    source=str(val_images),
    save=True,
    conf=0.5
)
print("Predicciones guardadas en:", results[0].save_dir)

Generando predicciones de validación...


NameError: name 'model' is not defined

In [ ]:
model.export(format="torchscript")